# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook guides you through loading and exploring the FAIR<sup>2</sup> dataset package using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a Croissant schema and available at: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and explore its structure with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant URL for the FAIR^2 dataset
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load metadata using mlcroissant
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Let's review available **record sets** and their structure. Each record set, field, and column is referenced by its `@id` as defined in the Croissant schema.

> **Note:** If you want additional information about the dataset's entities, you can inspect the metadata object for record sets (`record_set`), fields, and their attributes.

In [ ]:
# List available record sets with their @id and fields
print("Available Record Sets (by @id):\n")
record_sets = getattr(metadata, 'record_set', [])  # Fallback in case there are none
if len(record_sets) == 0:
    print("No record sets defined in the metadata.")
else:
    for rec in record_sets:
        rec_id = getattr(rec, '@id', None)
        rec_name = getattr(rec, 'name', 'N/A')
        print(f"- Record Set @id: {rec_id}; Name: {rec_name}")
        fields = getattr(rec, 'field', [])
        print("    Fields:")
        for field in fields:
            field_id = getattr(field, '@id', None)
            field_name = getattr(field, 'name', 'N/A')
            print(f"      • {field_id}: {field_name}")

## 3. Data Extraction
Let's extract data from record sets using their `@id`. You'll find record set and field IDs in the previous section output.

In [ ]:
# Identify all available record_set @id
# If the metadata.record_set is empty, let's try to parse the fileObject records (as commonly mass spec datasets store their tables there)
record_set_ids = []
if hasattr(metadata, 'record_set') and metadata.record_set:
    for rec in metadata.record_set:
        rec_id = getattr(rec, '@id', None)
        if rec_id:
            record_set_ids.append(rec_id)
else:
    # As a fallback: mlcroissant allows listing all record sets directly
    record_set_ids = [x['@id'] for x in dataset.list_recordsets()]

print("Record Sets IDs found in the dataset:")
print(record_set_ids)

# Load each record set into a pandas DataFrame
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

if record_set_ids:
    print(f"\nColumns in '{record_set_ids[0]}':")
    print(dataframes[record_set_ids[0]].columns.tolist())
    display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)
Let's filter, normalize, and group the data using key fields. All columns referenced are by their exact names (`@id` values) based on the record set schema.

> If you are not sure about the correct field `@id` to use as numeric or grouping field, check the columns output from the previous code cell.

In [ ]:
# Example EDA: filter and normalize a numeric field in the first record set
if record_set_ids:
    df = dataframes[record_set_ids[0]]

    # Heuristically pick a numeric field (e.g., 'cr:Peptide_Count' or similar) for demonstration
    numeric_field_id = None
    for c in df.columns:
        # Use a common identifier for peptide count or molecular weight
        if 'count' in c.lower() or 'mw' in c.lower() or 'abundance' in c.lower() or 'coverage' in c.lower():
            numeric_field_id = c
            break
    if numeric_field_id is None:
        numeric_field_id = df.columns[0]  # fallback to first column

    print(f"Using '{numeric_field_id}' as a numeric field for analysis.")

    # Try convert to numeric incase it's a string column
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    threshold = df[numeric_field_id].quantile(0.75)  # filter to upper quartile as example
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a textual or categorical field if available
    group_field = None
    for c in df.columns:
        if ('group' in c.lower() or 'category' in c.lower() or 'sample' in c.lower() or 'type' in c.lower()) and c != numeric_field_id:
            group_field = c
            break
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
        print(f"Grouped mean of {numeric_field_id} by {group_field}:")
        display(grouped_df.head())
    else:
        print("Could not identify a grouping field in this record set.")
else:
    print('No record sets available for EDA.')

## 5. Visualization
Let's visualize the distribution of the chosen numeric field in the filtered data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_field_id in filtered_df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id].dropna(), bins=20, kde=True, color='royalblue')
    plt.title(f"Distribution of {numeric_field_id} (filtered)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
else:
    print('Did not find suitable data for visualization.')

## 6. Conclusion
In this notebook, we used the [`mlcroissant`](https://github.com/mlcommons/croissant) library to load and explore the FAIR<sup>2</sup> dataset of proteins detected in extracellular vesicles from human mast cells. 

- We identified and loaded record sets and their fields by their unique `@id` as defined in the Croissant schema.
- We demonstrated filtering, normalization, and grouping using the dataset's field IDs.
- We visualized the distribution of abundance/count-based fields.

This workflow can be adapted to other Croissant-based datasets to ensure robust, FAIR methodology and analytics.
